# AIC 2026 - Kiểm Thử Captioning Video Shot Với Qwen3-VL-8B-Instruct Trên Kaggle

Notebook này được thiết kế để kiểm thử chuyên sâu mô hình **`Qwen/Qwen3-VL-8B-Instruct`** trên **12 Testcases Chuẩn** của dự án AIC 2026:
- **Mô hình:** `Qwen/Qwen3-VL-8B-Instruct` (~16.5 GB weights) - Phiên bản kích thước lớn hơn, suy luận ngữ cảnh sâu hơn, bóc tách thực thể và công thức/OCR với độ chính xác cao nhất.
- **Phần cứng khuyến nghị trên Kaggle:** Chọn **GPU T4 x2 (Dual T4 - 32GB VRAM)** hoặc **GPU P100 (16GB)**. Với cơ chế `device_map="auto"`, model sẽ tự động phân bổ cân bằng trên 2 GPU (~8.5 GB mỗi GPU), chạy hoàn toàn 100% trên VRAM mà không bị nghẽn CPU offload!
- **12 Testcases Đa Dạng:** Bao gồm các tình huống thử thách cao nhất: Ẩm thực (chế biến, nguyên liệu tươi), Lễ hội lân sư rồng (hoa văn, chuyển động), Đua xe đạp (đọc số liệu điện tử, cự ly km theo thời gian thực), và Bài giảng truyền hình (đọc công thức vật lý, ký hiệu toán học).

--- 
## BƯỚC 1: Kiểm Tra Cấu Hình GPU Trên Kaggle

In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"Số lượng GPU phát hiện: {device_count}")
for i in range(device_count):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)} | VRAM: {round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2)} GB")
if device_count >= 2:
    print("🔥 Tuyệt vời! Bạn đang dùng Dual GPU, model 8B sẽ được tự động chia đều để chạy với tốc độ tối đa!")
print("=" * 60)

--- 
## BƯỚC 2: CẤU HÌNH DATASET & DANH SÁCH 12 TESTCASES CHUẨN

Notebook sẽ tự động dò tìm đường dẫn chứa 12 video shot testcases trong thư mục `/kaggle/input`.

In [ ]:
import os
import re
import json
from datetime import datetime

# ==============================================================================
# 🎯 1. CẤU HÌNH ĐƯỜNG DẪN DATASET TRÊN KAGGLE (ĐỂ TRỐNG "" ĐỂ TỰ ĐỘNG DÒ TÌM):
# ==============================================================================
VIDEO_DIR = ""       # Folder chứa video shot (.mp4)
MEDIA_INFO_DIR = ""  # Folder chứa metadata {video_id}.json
ASR_DIR = ""         # Folder chứa {video_id}.json lời thoại

# File kết quả đầu ra
OUTPUT_DIR = "/kaggle/working"
OUTPUT_JSONL = os.path.join(OUTPUT_DIR, "qwen3vl_8b_captions_12_testcases.jsonl")
RESUME_CHECKPOINT = True

# ==============================================================================
# 📋 2. DANH SÁCH 12 TESTCASES CHUẨN TỪ LOCAL:
# ==============================================================================
TARGET_TEST_FILENAMES = [
    "L26_V480_shot_0060_03m04s_03m08s.mp4",  # 1. Ẩm thực: Chảo gừng dứa cá dứa
    "L24_V013_shot_0063_07m06s_07m12s.mp4",  # 2. Lễ hội: Con lân đỏ cận cảnh
    "L25_V060_shot_0197_18m52s_19m00s.mp4",  # 3. Giáo dục: Bài giảng Địa lý dân cư
    "L26_V254_shot_0004_00m12s_00m19s.mp4",  # 4. Ẩm thực: Nghêu kho tộ
    "L23_V017_shot_0017_01m41s_01m43s.mp4",  # 5. Thể thao: Đua xe đạp cự ly 0.8km
    "L24_V013_shot_0062_06m59s_07m06s.mp4",  # 6. Lễ hội: Con lân đỏ trung cảnh
    "L23_V017_shot_0016_01m36s_01m41s.mp4",  # 7. Thể thao: Đua xe đạp cự ly 0.9km
    "L23_V010_shot_0003_00m20s_00m25s.mp4",  # 8. Thể thao: Đua xe đạp Martin Laas chặng 10
    "L26_V074_shot_0007_00m28s_00m32s.mp4",  # 9. Ẩm thực: Lẩu Thái và hoa (toàn cảnh bàn)
    "L26_V074_shot_0005_00m16s_00m23s.mp4",  # 10. Ẩm thực: Lẩu Thái và hoa (cận hoa & tôm)
    "L26_V074_shot_0006_00m23s_00m28s.mp4",  # 11. Ẩm thực: Lẩu Thái và hoa (hoa & thịt heo)
    "L25_V014_shot_0179_15m28s_15m30s.mp4",  # 12. Giáo dục: Bài giảng Vật lý công thức lambda
]

# ==============================================================================
# 📂 3. TỰ ĐỘNG DÒ TÌM THƯ MỤC SIÊU TỐC TRONG /kaggle/input (0.01 GIÂY):
# ==============================================================================
input_base = "/kaggle/input"

def find_dir_fast(base_path, target_names):
    """Dò tìm nhanh thư mục mà KHÔNG duyệt đệ quy vào 873 subfolder video."""
    if not os.path.exists(base_path):
        return ""
    for root, dirs, _ in os.walk(base_path):
        # Pruning: Loại bỏ duyệt sâu vào các folder video Lxx_Vxxx để tránh nghẽn I/O trên Kaggle FUSE
        dirs[:] = [d for d in dirs if not (d.startswith('L') and '_V' in d)]
        for d in dirs:
            if d.lower() in target_names:
                return os.path.join(root, d)
    return ""

if not VIDEO_DIR or not os.path.exists(VIDEO_DIR):
    VIDEO_DIR = find_dir_fast(input_base, ["cut_videos", "cut_video", "video_shots", "videos"])
    if VIDEO_DIR:
        print(f"💡 [Tự động phát hiện] Gán VIDEO_DIR = {VIDEO_DIR}")

if not MEDIA_INFO_DIR or not os.path.exists(MEDIA_INFO_DIR):
    MEDIA_INFO_DIR = find_dir_fast(input_base, ["media-info", "media_info", "mediainfo"])
    if MEDIA_INFO_DIR:
        print(f"💡 [Tự động phát hiện] Gán MEDIA_INFO_DIR = {MEDIA_INFO_DIR}")

if not ASR_DIR or not os.path.exists(ASR_DIR):
    ASR_DIR = find_dir_fast(input_base, ["asr"])
    if ASR_DIR:
        print(f"💡 [Tự động phát hiện] Gán ASR_DIR = {ASR_DIR}")

# ==============================================================================
# 🔍 4. TÌM KIẾM TRỰC TIẾP 12 TESTCASES (TRUY XUẤT THẲNG THEO VIDEO_ID TRONG 0.001s):
# ==============================================================================
test_shot_paths = []
print("=" * 75)
print("🔍 TÌM KIẾM 12 TESTCASES TRONG DATASET:")

for idx, fname in enumerate(TARGET_TEST_FILENAMES, 1):
    parts = fname.split('_')
    vid = f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else ""
    
    # 1. Kiểm tra trực tiếp theo cấu trúc chuẩn: cut_videos/{video_id}/{fname}
    direct_path = os.path.join(VIDEO_DIR, vid, fname) if VIDEO_DIR and vid else ""
    flat_path = os.path.join(VIDEO_DIR, fname) if VIDEO_DIR else ""
    
    found_path = ""
    if direct_path and os.path.exists(direct_path):
        found_path = direct_path
    elif flat_path and os.path.exists(flat_path):
        found_path = flat_path
    else:
        # Fallback dò tìm chỉ trong folder của video_id đó nếu cần
        for root, dirs, files in os.walk(input_base):
            dirs[:] = [d for d in dirs if not (d.startswith('L') and '_V' in d and d != vid)]
            if fname in files:
                found_path = os.path.join(root, fname)
                break
                
    if found_path:
        test_shot_paths.append(found_path)
        print(f"  [{idx:02d}/12] ✅ Tìm thấy: {fname}")
    else:
        print(f"  [{idx:02d}/12] ❌ Không thấy: {fname}")

print(f"\n👉 Tổng số testcase sẵn sàng chạy: {len(test_shot_paths)}/{len(TARGET_TEST_FILENAMES)}")
print(f"👉 File lưu kết quả: {OUTPUT_JSONL}")
print("=" * 75)

--- 
## BƯỚC 3: Cài Đặt & Nâng Cấp Thư Viện Cần Thiết

In [ ]:
import torch
cuda_tag = f"cu{torch.version.cuda.replace('.', '')}" if (torch.cuda.is_available() and torch.version.cuda) else "cu121"
!pip install -q --upgrade "transformers>=4.49.0" "accelerate>=1.2.0" "qwen-vl-utils[decord]>=0.0.8" huggingface_hub
!pip install -q --upgrade --force-reinstall torchvision --extra-index-url https://download.pytorch.org/whl/{cuda_tag}

import transformers
import qwen_vl_utils
import torchvision
print("✅ Đã nâng cấp thư viện thành công:")
print(f"  - PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"  - Torchvision: {torchvision.__version__}")
print(f"  - Transformers: {transformers.__version__}")
print(f"  - Qwen-VL-Utils: {qwen_vl_utils.__file__}")

--- 
## BƯỚC 4: Cấu Hình Prompt Chuẩn Hóa (Motion-Centric & Fact-Color)

In [ ]:
SYSTEM_PROMPT = (
    "Bạn là người hỗ trợ phân tích thị giác và lập chỉ mục dữ liệu video phục vụ hệ thống tìm kiếm (Video Retrieval). "
    "Mỗi đoạn video là MỘT CÚ MÁY ĐƠN LIÊN TỤC (single continuous shot, không có chuyển cảnh). "
    "Nhiệm vụ của bạn là quan sát diễn biến, bóc tách thực thể, màu sắc, số lượng và chi tiết thị giác "
    "một cách tự nhiên, chính xác và giàu thông tin nhất."
)

def build_user_prompt(asr_text: str, video_title: str = "") -> str:
    """Tạo prompt tinh gọn 2 mục, nạp động tiêu đề video metadata và ngữ cảnh ASR."""
    asr_context = (
        f'"{asr_text.strip()}"'
        if asr_text and asr_text.strip()
        else "(Không có lời thoại / Video chỉ có âm thanh hiện trường hoặc nhạc nền)"
    )

    title_block = ""
    if video_title and video_title.strip():
        title_block = f'\n[TIÊU ĐỀ VIDEO / CHƯƠNG TRÌNH]:\n"{video_title.strip()}"\n'

    return f"""Hãy phân tích cú máy video này phục vụ tìm kiếm thông tin:
{title_block}
[LỜI THOẠI ASR]:
{asr_context}

HƯỚNG DẪN MÔ TẢ:
1. Trọng tâm chuyển động: Tập trung miêu tả hành động chính, người hoặc vật đang chuyển động và các sự vật bị tác động chuyển động theo.
2. Màu sắc & Số lượng (Kể cả đặc điểm hiển nhiên): Luôn chủ động gắn kèm màu sắc và số lượng (hoặc khoảng ước lượng nếu có) cho các thực thể/danh từ, ngay cả khi đó là thuộc tính hiển nhiên (ví dụ: củ dền màu đỏ/tím, cọng sả màu xanh/vàng, lòng đỏ trứng màu cam/vàng). Nếu chưa chắc chắn về tên vật thể hoặc màu sắc, dùng 'hoặc' (hoặc '/') để dự đoán gần đúng.
3. Tích hợp Tiêu đề, OCR & ASR: Kết hợp tiêu đề video, lời thoại ASR và đọc văn bản chữ quan trọng trên màn hình (nếu có) để nêu đúng tên người, sự kiện, địa điểm, tên món ăn hoặc tiết mục.

XUẤT ĐẦU RA THEO ĐÚNG 2 MỤC SAU (KHÔNG THÊM LỜI DẪN):
- Chi tiết thị giác & Thực thể: (Miêu tả tự nhiên hành động, chuyển động của nhân vật, trang phục, màu sắc, số lượng nếu có, đạo cụ, đồ vật chính và chữ hiển thị trên màn hình nếu có)
- Góc máy & Không gian: (Góc quay cận cảnh/trung cảnh/toàn cảnh, chuyển động camera nếu có, không gian trong nhà/ngoài trời)

VÍ DỤ MẪU:
Ví dụ 1:
- Chi tiết thị giác & Thực thể: Một đôi đũa đang đảo 4 hoặc 6 miếng thịt gà (hoặc thịt ếch) màu vàng trong chảo màu đỏ đặt trên bếp gas đang cháy lửa xanh. Trong chảo có 2-3 cọng sả màu xanh hoặc vàng đang được đảo cùng thịt. Phía dưới có dòng chữ tiêu đề hiển thị tên món ăn.
- Góc máy & Không gian: Góc quay cận cảnh chảo và bếp, sau đó nâng dần lên góc nhìn từ trên xuống; không gian trong nhà bếp.

Ví dụ 2:
- Chi tiết thị giác & Thực thể: Cuộc đua xe đạp ở đoạn về đích, người dẫn đầu mặc áo xanh hoặc vàng buông hai tay khỏi ghi đông để ăn mừng trước vạch đích. Phía sau có 2 người mặc áo đỏ và trắng lần lượt bám đuổi. Bảng điện tử ghi 'CHẶNG 17 - 0.8 Km'.
- Góc máy & Không gian: Góc quay toàn cảnh từ trên cao nhìn xuống đường phố rộng ngoài trời, camera lia bám theo đoàn đua.
"""

print("✅ Đã cấu hình Prompt chuẩn hóa 2 mục Motion-Centric & Fact-Color!")

--- 
## BƯỚC 5: Nạp Model Qwen3-VL-8B-Instruct Với device_map='auto'

Model **`Qwen/Qwen3-VL-8B-Instruct`** có kích thước ~16.5 GB weights. Trên Kaggle GPU T4 x2 (Dual T4 - 32GB VRAM), mô hình sẽ tự động chia đều lên 2 GPU, nạp 100% trong VRAM không cần offload CPU!

In [ ]:
import time
from transformers import AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MIN_PIXELS = 128 * 28 * 28   # 100,352 pixels
MAX_PIXELS = 360 * 640       # 230,400 pixels

print(f"[1/2] Đang nạp Processor cho model: {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS
)

print(f"[2/2] Đang nạp Model weights với dtype=bfloat16, device_map='auto'...")
start_load = time.time()
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

candidate_classes = [
    "Qwen3VLForConditionalGeneration",
    "AutoModelForMultimodalLM",
    "AutoModelForImageTextToText"
]

model = None
loaded_class_name = None

for class_name in candidate_classes:
    if hasattr(transformers, class_name):
        try:
            cls = getattr(transformers, class_name)
            print(f"  Thử nạp với class: {class_name}...")
            model = cls.from_pretrained(
                MODEL_ID,
                torch_dtype=dtype,
                device_map="auto"
            )
            loaded_class_name = class_name
            print(f"  => ✅ NẠP THÀNH CÔNG với class: {class_name} ({round(time.time() - start_load, 2)}s)")
            break
        except Exception as e:
            print(f"  Class {class_name} thất bại: {e}")
            continue

if model is None:
    raise RuntimeError(f"Không thể nạp model {MODEL_ID}!")

model.eval()

print("\n--- THỐNG KÊ PHÂN BỔ VRAM SAU KHI NẠP ---")
for i in range(torch.cuda.device_count()):
    alloc_gb = round(torch.cuda.memory_allocated(i) / (1024**3), 2)
    res_gb = round(torch.cuda.memory_reserved(i) / (1024**3), 2)
    print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): Allocated = {alloc_gb} GB | Reserved = {res_gb} GB")
if hasattr(model, "hf_device_map"):
    print(f"  [Device Map]: Các layers được phân bổ trên thiết bị: {set(model.hf_device_map.values())}")
print("-------------------------------------------")

--- 
## BƯỚC 6: Các Tiện Ích Trích Xuất Metadata & Hàm Suy Luận

In [ ]:
import os
import re
import json
from qwen_vl_utils import process_vision_info

def parse_shot_filename(filename: str):
    """Trích xuất video_id, shot_idx, start_sec, end_sec từ tên file shot."""
    stem = os.path.splitext(os.path.basename(filename))[0]
    parts = stem.split('_')
    if len(parts) >= 6:
        video_id = f"{parts[0]}_{parts[1]}"
        try:
            shot_idx = int(parts[3])
        except Exception:
            shot_idx = 0
        def to_sec(ts: str) -> float:
            h_m = re.search(r'(\d+)h', ts)
            m_m = re.search(r'(\d+)m', ts)
            s_m = re.search(r'(\d+)s', ts)
            hours = int(h_m.group(1)) if h_m else 0
            mins = int(m_m.group(1)) if m_m else 0
            secs = int(s_m.group(1)) if s_m else 0
            return float(hours * 3600 + mins * 60 + secs)
        start_sec = to_sec(parts[-2])
        end_sec = to_sec(parts[-1])
        return video_id, shot_idx, start_sec, end_sec
    return stem, 0, 0.0, 0.0


def get_video_title(video_id: str, media_info_dir: str) -> str:
    """Đọc file JSON metadata trong media-info tương ứng với video_id và lấy trường title."""
    if not media_info_dir or not os.path.exists(media_info_dir):
        return ""
    info_path = os.path.join(media_info_dir, f"{video_id}.json")
    if not os.path.exists(info_path):
        return ""
    try:
        with open(info_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data.get("title", "").strip()
    except Exception:
        return ""


def get_asr_context(video_id: str, shot_start: float, shot_end: float, asr_dir: str):
    """Lọc các segment lời thoại ASR giao thoa với khoảng thời gian của shot."""
    if not asr_dir or not os.path.exists(asr_dir):
        return "", 0
    asr_path = os.path.join(asr_dir, f"{video_id}.json")
    if not os.path.exists(asr_path):
        return "", 0
    try:
        with open(asr_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        segments = data if isinstance(data, list) else data.get("segments", [])
        matched_texts = []
        for seg in segments:
            s_start = seg.get("start", 0.0)
            s_end = seg.get("end", 0.0)
            text = seg.get("text", "").strip()
            if not text:
                continue
            if max(s_start, shot_start) < min(s_end, shot_end):
                matched_texts.append(text)
        full_text = " ".join(matched_texts).strip()
        return full_text, len(matched_texts)
    except Exception:
        return "", 0


def caption_single_shot(
    model,
    processor,
    video_path: str,
    asr_text: str = "",
    video_title: str = "",
    fps: float = 1.0,
    max_new_tokens: int = 350,
    repetition_penalty: float = 1.05,
):
    """Sinh caption cho một video shot duy nhất sử dụng Qwen3-VL-8B."""
    start_time = time.time()
    user_prompt = build_user_prompt(asr_text, video_title)
    video_uri = f"file://{os.path.abspath(video_path)}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_uri,
                    "fps": fps,
                },
                {"type": "text", "text": user_prompt},
            ],
        },
    ]

    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    vision_out = process_vision_info(messages, return_video_kwargs=True)
    if isinstance(vision_out, tuple) and len(vision_out) == 3:
        image_inputs, video_inputs, video_kwargs = vision_out
    elif isinstance(vision_out, tuple) and len(vision_out) == 2:
        image_inputs, video_inputs = vision_out
        video_kwargs = {}
    else:
        image_inputs, video_inputs = None, vision_out
        video_kwargs = {}

    if isinstance(video_kwargs, dict):
        if "fps" in video_kwargs and isinstance(video_kwargs["fps"], (list, tuple)):
            if len(video_kwargs["fps"]) > 0:
                video_kwargs["fps"] = float(video_kwargs["fps"][0])
            else:
                video_kwargs.pop("fps", None)

    inputs = processor(
        text=[prompt_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        **video_kwargs
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=repetition_penalty
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = generated_ids[:, input_len:]
    caption = processor.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    return {
        "caption": caption,
        "infer_time_sec": round(time.time() - start_time, 2),
        "input_tokens": int(input_len),
        "generated_tokens": int(generated_tokens.shape[1]),
    }

print("✅ Đã khởi tạo các hàm tiện ích và inference cho Qwen3-VL-8B!")

--- 
## BƯỚC 7: Chạy Suy Luận Cho 12 Testcases (Qwen3-VL-8B-Instruct)

In [ ]:
shots_to_process = test_shot_paths if 'test_shot_paths' in globals() and test_shot_paths else []

print(f"🚀 BẮT ĐẦU CHẠY KIỂM THỬ QWEN3-VL-8B TRÊN {len(shots_to_process)} TESTCASES")
print(f"   - File kết quả: {OUTPUT_JSONL}\n")

# Tải các shot đã hoàn thành trước đó (nếu resume)
done_files = set()
if RESUME_CHECKPOINT and os.path.exists(OUTPUT_JSONL):
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    r = json.loads(line)
                    done_files.add(r.get("shot_file"))
                except Exception:
                    pass

actual_todo = [p for p in shots_to_process if os.path.basename(p) not in done_files]
print(f"   - Đã làm trước đó:     {len(done_files)} shots (Bỏ qua)")
print(f"   - Thực tế cần chạy:    {len(actual_todo)} shots\n")

if len(actual_todo) == 0:
    print("🎉 TẤT CẢ CÁC TESTCASES ĐÃ HOÀN TẤT!")
else:
    session_start_time = time.time()
    for idx, video_path in enumerate(actual_todo, 1):
        filename = os.path.basename(video_path)
        video_id, shot_idx, start_sec, end_sec = parse_shot_filename(filename)
        duration = round(end_sec - start_sec, 2)
        
        elapsed = time.time() - session_start_time
        avg_speed = elapsed / max(1, idx - 1) if idx > 1 else 15.0
        eta_mins = round(((len(actual_todo) - idx) * avg_speed) / 60, 1)
        
        print("-" * 80)
        print(f"[{idx}/{len(actual_todo)}] ({idx/len(actual_todo)*100:.1f}%) | ETA: ~{eta_mins}m | SHOT: {filename} ({duration}s)")
        
        # Nạp động metadata title và ASR
        video_title = get_video_title(video_id, MEDIA_INFO_DIR)
        asr_text, seg_count = get_asr_context(video_id, start_sec, end_sec, ASR_DIR)
        
        if video_title:
            print(f"  - Tiêu đề: \"{video_title}\"")
        if asr_text:
            print(f"  - Lời thoại ASR: \"{asr_text[:90]}...\" ({seg_count} segments)")
            
        try:
            res = caption_single_shot(
                model=model,
                processor=processor,
                video_path=video_path,
                asr_text=asr_text,
                video_title=video_title,
            )
            
            print(f"  - Thời gian: {res['infer_time_sec']}s | Tokens: In={res['input_tokens']}, Out={res['generated_tokens']}")
            print("  - CAPTION (Qwen3-VL-8B):")
            for line in res["caption"].split("\n"):
                if line.strip():
                    print(f"    {line}")
                    
            target_idx = TARGET_TEST_FILENAMES.index(filename) + 1 if filename in TARGET_TEST_FILENAMES else idx
            record = {
                "test_case_idx": target_idx,
                "video_id": video_id,
                "shot_file": filename,
                "shot_idx": shot_idx,
                "shot_start_sec": start_sec,
                "shot_end_sec": end_sec,
                "duration_sec": duration,
                "video_title": video_title,
                "asr_segment_count": seg_count,
                "asr_context": asr_text,
                "caption": res["caption"],
                "model_id": MODEL_ID,
                "infer_time_sec": res["infer_time_sec"],
                "input_tokens": res["input_tokens"],
                "generated_tokens": res["generated_tokens"],
                "timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
            }
            
            with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                f.flush()
                
        except Exception as e:
            print(f"  ❌ Lỗi khi xử lý shot {filename}: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
    print("\n" + "=" * 80)
    print(f"✅ HOÀN TẤT TOÀN BỘ 12 TESTCASES CHO QWEN3-VL-8B!")
    print(f"📁 File kết quả lưu tại: {OUTPUT_JSONL}")
    print("=" * 80)

--- 
## BƯỚC 8: Bảng Thống Kê Kết Quả Caption 8B

In [ ]:
import pandas as pd

if os.path.exists(OUTPUT_JSONL):
    records = []
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    
    if records:
        df = pd.DataFrame(records)
        cols = [c for c in ["test_case_idx", "video_id", "shot_file", "duration_sec", "infer_time_sec", "generated_tokens"] if c in df.columns]
        display(df[cols])
        print(f"\n📊 TỔNG KẾT KIỂM THỬ QWEN3-VL-8B:")
        print(f"  - Tổng số testcases thành công: {len(df)}/12")
        print(f"  - Thời gian suy luận trung bình: {df['infer_time_sec'].mean():.2f} giây/shot")
        print(f"  - Số tokens sinh ra trung bình:  {df['generated_tokens'].mean():.1f} tokens/shot")
else:
    print("Chưa có file kết quả. Hãy chạy Bước 7!")